# Regional Obligations: Bronze → Silver

Cleans `pondolens.dev_bronze.regional_expenditure_obligations` and merges it into `pondolens.dev_silver.regional_obligations`.

**Silver columns:** `obligation_id | year | category | particulars | regions | expenditure_amount | processing_run_id | processed_at`

**Rules**
- **Business key:** `year, category, particulars, regions`. The MERGE matches Bronze to Silver on this key.
- `obligation_id`: same record, same ID, even when its amount is updated. Only new records get an ID: last allocated ID (2,000,000 when empty) + `ROW_NUMBER()`, applied to new records only.
- Reusable: rerun or refresh anytime. New Bronze data is inserted, changed amounts are updated, nothing is duplicated and existing IDs never change.
- `expenditure_amount`: `-` becomes 0, blanks stay NULL, thousand pesos are converted to pesos (× 1000).
- Each run gets one `processing_run_id` and one `processed_at`. Only new or changed rows get stamped, so unchanged rows keep the run that last touched them.
- The notebook stops (ASSERT_TRUE) on missing keys, duplicate keys, unknown units, unparseable amounts or out-of-range IDs.

Run the cells top to bottom.

## 1. Create the Silver table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.regional_obligations (
  obligation_id INT NOT NULL,
  year INT NOT NULL,
  category STRING NOT NULL,
  particulars STRING,
  regions STRING NOT NULL,
  expenditure_amount DECIMAL(20,2),
  processing_run_id STRING NOT NULL,
  processed_at TIMESTAMP NOT NULL,
  CONSTRAINT regional_obligations_pk PRIMARY KEY (obligation_id)
)
USING DELTA
COMMENT 'Regional expenditure obligations (actual obligations) in pesos. Grain: year, category, particulars, regions.';

## 2. Run metadata

One ID and one timestamp for the whole run, so every row written in this run shares the same values.

In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_run_id STRING;
DECLARE OR REPLACE VARIABLE v_run_ts TIMESTAMP;

SET VAR
  v_run_id = REPLACE(UUID(), '-', ''),
  v_run_ts = CURRENT_TIMESTAMP();

SELECT v_run_id AS processing_run_id, v_run_ts AS processed_at;

## 3. Clean Bronze into a staging view

- Trims text and collapses repeated spaces; blank text becomes NULL.
- Standardizes region spellings and converts them to uppercase (e.g. `Region-IVA`, `Region IV-A` → `REGION IVA`; `Nationwide` → `NATIONWIDE`).
- If Bronze holds more than one ingestion of the same record, keeps only the latest ingestion.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW regional_obligation_stage AS
WITH cleaned AS (
  SELECT
    TRY_CAST(TRIM(CAST(year AS STRING)) AS INT) AS year,
    NULLIF(REGEXP_REPLACE(TRIM(category), r'\s+', ' '), '') AS category,
    NULLIF(REGEXP_REPLACE(TRIM(particulars), r'\s+', ' '), '') AS particulars,

    -- Standardize Region-IVA and Region IV-A to REGION IVA; all regions in uppercase.
    UPPER(CASE
      WHEN TRIM(regions) RLIKE r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$'
      THEN CONCAT(
        'Region ',
        UPPER(REGEXP_EXTRACT(TRIM(regions), r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$', 1)),
        UPPER(REGEXP_EXTRACT(TRIM(regions), r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$', 2))
      )
      WHEN LOWER(TRIM(regions)) = 'nationawide' THEN 'Nationwide'
      ELSE NULLIF(REGEXP_REPLACE(TRIM(regions), r'\s+', ' '), '')
    END) AS regions,

    LOWER(TRIM(unit_amount)) AS unit_amount,
    NULLIF(REPLACE(TRIM(CAST(expenditure_amount AS STRING)), ',', ''), '') AS amount_text,
    _ingested_at
  FROM pondolens.dev_bronze.regional_expenditure_obligations
),
latest AS (
  -- Keep the latest ingestion of each record. RANK (not ROW_NUMBER) keeps ties,
  -- so conflicting values inside the same ingestion are caught by the checks below.
  SELECT *
  FROM (
    SELECT
      *,
      RANK() OVER (
        PARTITION BY year, category, particulars, regions
        ORDER BY _ingested_at DESC
      ) AS ingest_rank
    FROM cleaned
  )
  WHERE ingest_rank = 1
)
SELECT DISTINCT
  year,
  category,
  particulars,
  regions,
  unit_amount,
  amount_text,

  -- Dashes become zero; blanks stay null. Convert thousand pesos to pesos.
  CASE
    WHEN amount_text IN ('-', '–', '—') THEN CAST(0 AS DECIMAL(20,2))
    WHEN amount_text IS NULL THEN CAST(NULL AS DECIMAL(20,2))
    WHEN unit_amount = 'thousand pesos'
      THEN CAST(TRY_CAST(amount_text AS DECIMAL(20,2)) * 1000 AS DECIMAL(20,2))
    WHEN unit_amount = 'pesos'
      THEN TRY_CAST(amount_text AS DECIMAL(20,2))
  END AS expenditure_amount
FROM latest;

## 4. Data quality checks on the staged data

In [0]:
%sql
-- Each ASSERT_TRUE stops the run with its own message if it fails.
SELECT
  ASSERT_TRUE(COUNT(*) > 0,
    'Bronze returned no rows.') AS has_rows,
  ASSERT_TRUE(COUNT(*) = COUNT(year) AND COUNT(*) = COUNT(category) AND COUNT(*) = COUNT(regions),
    'Missing year, category or regions.') AS keys_present,
  ASSERT_TRUE(COUNT_IF(year NOT BETWEEN 2000 AND 2100) = 0,
    'Year outside 2000-2100.') AS valid_year,
  ASSERT_TRUE(COUNT_IF(unit_amount IS NULL OR unit_amount NOT IN ('thousand pesos', 'pesos')) = 0,
    'Unknown unit_amount; update the conversion rule.') AS known_unit,
  ASSERT_TRUE(COUNT_IF(amount_text IS NOT NULL AND expenditure_amount IS NULL) = 0,
    'Some expenditure_amount values could not be parsed as numbers.') AS amounts_parsed,
  ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT STRUCT(year, category, particulars, regions)),
    'Conflicting records: same year/category/particulars/regions with different values.') AS unique_key
FROM regional_obligation_stage;

## 5. Check the existing Silver table

In [0]:
%sql
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT obligation_id)
  AND COUNT(*) = COUNT(DISTINCT STRUCT(year, category, particulars, regions))
  AND COUNT_IF(obligation_id <= 2000000) = 0,
  'Silver has duplicate IDs, duplicate keys or IDs at or below 2000000.'
)
FROM pondolens.dev_silver.regional_obligations;

## 6. Assign obligation_id to new records only

**Business key:** `year, category, particulars, regions`

- A record is *new* when its business key is not yet in Silver.
- `ROW_NUMBER()` runs on new records only and continues from the last allocated ID (2000000 when Silver is empty).
- Existing records are not renumbered, so their IDs never change, even when their amount is updated.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW regional_obligation_new_ids AS
WITH new_records AS (
  -- Business keys in Bronze that are not yet in Silver.
  SELECT s.year, s.category, s.particulars, s.regions
  FROM regional_obligation_stage AS s
  LEFT ANTI JOIN pondolens.dev_silver.regional_obligations AS t
    ON s.year = t.year
    AND s.category = t.category
    AND s.particulars <=> t.particulars
    AND s.regions = t.regions
),
last_id AS (
  -- Last allocated ID; 2000000 when the table is empty.
  SELECT COALESCE(MAX(CAST(obligation_id AS BIGINT)), 2000000) AS last_number
  FROM pondolens.dev_silver.regional_obligations
)
SELECT
  n.year,
  n.category,
  n.particulars,
  n.regions,
  l.last_number + ROW_NUMBER() OVER (
    ORDER BY n.year, n.category, n.particulars, n.regions
  ) AS obligation_id
FROM new_records AS n
CROSS JOIN last_id AS l;

-- Ensure new IDs fit the INT column.
SELECT ASSERT_TRUE(
  COUNT_IF(obligation_id NOT BETWEEN 2000001 AND 2147483647) = 0,
  'New obligation_id values must be between 2000001 and 2147483647.'
)
FROM regional_obligation_new_ids;

## 7. Merge into Silver using the business key

- **Matched** (key already in Silver): update the amount only if it changed. The ID stays the same.
- **Not matched** (new key): insert with the new ID from step 6.

In [0]:
%sql
-- Attach the new IDs to the staged records. Existing records get NULL here,
-- and the MERGE never uses it for them, so their IDs stay unchanged.
CREATE OR REPLACE TEMP VIEW regional_obligation_merge_source AS
SELECT
  s.year,
  s.category,
  s.particulars,
  s.regions,
  s.expenditure_amount,
  n.obligation_id
FROM regional_obligation_stage AS s
LEFT JOIN regional_obligation_new_ids AS n
  ON s.year = n.year
  AND s.category = n.category
  AND s.particulars <=> n.particulars
  AND s.regions = n.regions;

MERGE INTO pondolens.dev_silver.regional_obligations AS t
USING regional_obligation_merge_source AS s
ON t.year = s.year
  AND t.category = s.category
  AND t.particulars <=> s.particulars
  AND t.regions = s.regions

-- Update changed amounts only; unchanged records are left untouched.
WHEN MATCHED
  AND NOT (t.expenditure_amount <=> s.expenditure_amount)
THEN UPDATE SET
  t.expenditure_amount = s.expenditure_amount,
  t.processing_run_id = v_run_id,
  t.processed_at = v_run_ts

WHEN NOT MATCHED THEN INSERT (
  obligation_id,
  year,
  category,
  particulars,
  regions,
  expenditure_amount,
  processing_run_id,
  processed_at
)
VALUES (
  CAST(s.obligation_id AS INT),
  s.year,
  s.category,
  s.particulars,
  s.regions,
  s.expenditure_amount,
  v_run_id,
  v_run_ts
);

## 8. Run summary

In [0]:
%sql
SELECT
  year,
  COUNT(*) AS records,
  COUNT_IF(processing_run_id = v_run_id) AS written_this_run,
  COUNT_IF(expenditure_amount IS NULL) AS null_amounts,
  MIN(obligation_id) AS min_id,
  MAX(obligation_id) AS max_id
FROM pondolens.dev_silver.regional_obligations
GROUP BY year
ORDER BY year;